# 02 · Test set

Flags crypto slang in every post, measures how common slang is, and draws a 300-post test set (150 slang-heavy + 150 plain) for hand-labeling. Once the posts are labeled by hand, it checks the labels and builds the final test set.

Sections 1–5 run before labeling. Section 6 is the labeling guide. Section 7 runs after the labeling sheet is filled in.

- **Inputs:**
  - `data/slang.csv`: crypto slang terms and their meanings
  - `data/processed/test_pool.csv`: the held-out pool from notebook 01
  - `data/interim.nosync/clean_posts.csv`: the full cleaned dataset from notebook 01 (not committed, so run notebook 01 first)
- **Outputs:**
  - `data/test_to_label.csv`: the labeling sheet, without author tags so the labels are blind
  - `data/test_hand_labeled.csv`: the final test set, with the hand label, hand slang flag and author tag for each post

## Setup

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd

# Let the notebook import our shared code from src/
ROOT = Path("..").resolve()
sys.path.append(str(ROOT))
from src.labels import LABELS

DATA = ROOT / "data"
SHEET_PATH = DATA / "test_to_label.csv"
TEST_PATH = DATA / "test_hand_labeled.csv"

SEED = 42
PER_GROUP = 50  # posts per (slang flag x author tag) group: 2 x 3 groups x 50 = 300
GROUPS = {1: "slang-heavy", 0: "plain"}

## 1. Load the slang list and the posts

`slang.csv` lists crypto slang terms with a plain-language meaning for each. The rules for what made the list:

- **In:** crypto-specific slang, written as text. Variants are listed separately (`hodl`, `hodling`, `hodler`), because matching is by whole word (section 2).
- **Left out on purpose:**
  - general internet slang (*lol*, *bro*);
  - common market words (*pump*, *dump*, *dip*, *ath*, *bull trap*);
  - the label words *bullish* and *bearish*, since a post that says "bullish" states its own label;
  - emojis, which are hard to read in sarcastic or meme posts.

In [2]:
slang = pd.read_csv(DATA / "slang.csv")
pool = pd.read_csv(DATA / "processed" / "test_pool.csv")
posts = pd.read_csv(DATA / "interim.nosync" / "clean_posts.csv")

print(f"{len(slang)} slang terms")
print(f"{len(pool):,} posts in the test pool, {len(posts):,} in the full cleaned dataset")
slang.sample(5, random_state=SEED)

118 slang terms
15,000 posts in the test pool, 1,254,911 in the full cleaned dataset


,term,meaning
56,shilled,Promoted a coin
89,copium,False comfort after a loss (cope + opium)
4,hodled,Held instead of selling
45,bagholders,People stuck holding a coin that has lost value
26,lfg,"""Let's f***ing go"": excitement, usually about ..."


## 2. Automatic slang flag

A post is flagged if it contains at least one term from the list as a **whole word or phrase**. This uses a *regular expression* (regex), a pattern that text is checked against:

- `\b` marks a word boundary, so `hodl` matches *"hodl"* and *"hodl!"* but not *"hodling"*. That's why variants are listed separately.
- Inside a phrase, `\s+` allows any spacing between the words, so *"diamond  hands"* with two spaces still matches.
- All the terms are joined with `|` (which means "or") into one pattern, so each post is scanned once.

The dataset author lowercased every post, and the terms are lowercase too, so upper and lower case don't need handling.

The flag is only a **first sort**. It can't tell slang from a literal use, like *"ser"* in a Swedish post, so the hand flag in the labeling sheet is final.

In [3]:
def term_pattern(term):
    """Regex for one slang term: its exact words, with any spacing between them."""
    return r"\s+".join(re.escape(word) for word in term.split())


SLANG_RE = re.compile(r"\b(?:" + "|".join(term_pattern(t) for t in slang["term"]) + r")\b")


def find_slang(text):
    """Slang terms found in a post, in order of appearance, without repeats."""
    found = []
    for match in SLANG_RE.findall(text):
        term = " ".join(match.split())  # "diamond  hands" -> "diamond hands"
        if term not in found:
            found.append(term)
    return found


find_slang("diamond hands, hodl till we moon 🚀"), find_slang("still hodling. wen lambo"), find_slang("holding until it goes up")

(['diamond hands', 'hodl', 'moon'], ['hodling', 'wen', 'lambo'], [])

In [4]:
pool["matched_terms"] = pool["text"].map(find_slang)
pool["auto_slang"] = (pool["matched_terms"].map(len) > 0).astype(int)

print(f"{pool['auto_slang'].mean():.1%} of test-pool posts are flagged")

7.0% of test-pool posts are flagged


## 3. How common is slang?

The test set is half slang-heavy on purpose, far more than in real posts. The **natural slang rate** shows the real share: the percentage of all 1.25M cleaned posts with at least one slang term. Because results are reported separately for slang-heavy and plain posts, this oversampling doesn't skew them.

The rate comes from the automatic flag, so it's an estimate. It counts literal matches like *"ser"* in Swedish and misses slang that isn't on the list. This cell takes about a minute.

In [5]:
posts["matched_terms"] = posts["text"].map(find_slang)
posts["auto_slang"] = posts["matched_terms"].map(len) > 0

rate = posts.groupby("label")["auto_slang"].mean().loc[LABELS]
rate["all posts"] = posts["auto_slang"].mean()
(rate * 100).round(1).rename("% of posts with slang").to_frame()

,% of posts with slang
label,
bullish,7.7
bearish,7.7
neutral,5.9
all posts,7.0


The most common terms, by number of posts they appear in:

In [6]:
posts["matched_terms"].explode().value_counts().head(10).rename("posts").to_frame()

,posts
matched_terms,
moon,11268
whales,9701
hodl,6979
lfg,4823
fomo,4122
fud,3916
whale,3710
buy the dip,3655
ape,2546


## 4. Draw the test set

50 posts are drawn at random from each of the 6 groups (automatic slang flag × author tag), from the held-out pool only, so none of them appear in training. Drawing by author tag keeps the sample from being mostly one class.

The final groups and classes come from the hand labels, which won't always match the automatic flag or the author's tag. So the final counts may move a little away from 150 / 150 and even classes. Section 7 reports them.

Posts available in the pool per group:

In [7]:
pd.crosstab(pool["auto_slang"].map(GROUPS), pool["label"], rownames=["automatic flag"], colnames=["author tag"])[LABELS]

author tag,bullish,bearish,neutral
automatic flag,,,
plain,4622,4615,4706
slang-heavy,378,385,294


In [8]:
sample = (
    pool.groupby(["auto_slang", "label"])
    .sample(n=PER_GROUP, random_state=SEED)
    .sample(frac=1, random_state=SEED)  # shuffle so slang and plain posts are mixed while labeling
    .reset_index(drop=True)
)

pd.crosstab(sample["auto_slang"].map(GROUPS), sample["label"], rownames=["automatic flag"], colnames=["author tag"])[LABELS]

author tag,bullish,bearish,neutral
automatic flag,,,
plain,50,50,50
slang-heavy,50,50,50


## 5. Write the labeling sheet

The sheet leaves out the author's tag, so the hand labels are blind. `matched_terms` shows which terms the automatic flag found, as a hint for the slang column.

The notebook **never overwrites** an existing sheet, so rerunning it can't erase labels. To start over, delete the file first.

In [9]:
sheet = sample[["post_id", "text"]].copy()
sheet["matched_terms"] = sample["matched_terms"].str.join("; ")
sheet["slang"] = ""
sheet["label"] = ""

if SHEET_PATH.exists():
    print(f"{SHEET_PATH.name} already exists, so it was left as is (it may hold labels).")
else:
    # utf-8-sig lets Excel show emoji correctly
    sheet.to_csv(SHEET_PATH, index=False, encoding="utf-8-sig")
    print(f"Wrote {SHEET_PATH.name}: {len(sheet)} posts to label")

test_to_label.csv already exists, so it was left as is (it may hold labels).


## 6. Labeling guide

Fill in two columns for each post in `data/test_to_label.csv`: `label` and `slang`. Leave the other columns as they are. Row order doesn't matter.

**`label`: the author's view of the price.** Use `bullish`, `bearish` or `neutral`.

| Label | When the author... | Example |
|---|---|---|
| `bullish` | expects or wants the price to rise, is buying or holding with confidence, or celebrates gains | *"loaded more, see you at 50k"* |
| `bearish` | expects the price to fall, is selling or shorting, warns others, or calls a coin a scam | *"this is heading to 20k, get out while you can"* |
| `neutral` | shows no clear lean: news, facts, plain questions, greetings, ads and spam | *"what time does the fed speak today?"* |

Edge cases:

- **Sarcasm:** label what the author means, not the literal words. *"great, another 20% drop, love this coin"* is bearish.
- **Mood vs. view:** an upset post can still be bullish. *"rekt again but buying more"* is bullish.
- **Questions:** neutral, unless the question clearly leans. *"anyone else loading up before the breakout?"* is bullish.
- **Mixed views:** go with the stronger lean. If neither is stronger, use neutral.
- **Ads, spam and bot posts:** neutral.
- **Can't tell what the post means:** neutral.

**`slang`: does the post use crypto slang?** Use `1` or `0`.

- **`1`:** the post uses at least one crypto slang term *as slang*, whether or not it's on the list (for example *"number go up"*). One term is enough.
- **`0`:** the post has no crypto slang, or the matched term is used literally (*"ser"* in a Swedish post, *"planet of the apes"*, *"send it to my wallet"*).
- `matched_terms` is only a hint. Follow the same rules as the list: general internet slang (*lol*, *bro*) and common market words (*pump*, *dip*, *ath*) don't count.

**Saving:** keep the file as a CSV. In Excel, use *File → Save As → CSV UTF-8 (Comma delimited)* so emoji survive. In Numbers, use *File → Export To → CSV*. Then run section 7.

## 7. Check the labels and build the test set

Run this after the sheet is filled in. If any post is missing a value or has a misspelled one, it stops and says which. Post text is taken from the test pool, not the sheet, so a spreadsheet app can't alter it.

In [10]:
labeled = pd.read_csv(SHEET_PATH, encoding="utf-8-sig")
labeled["label"] = labeled["label"].astype("string").str.strip().str.lower()

same_posts = len(labeled) == len(sample) and set(labeled["post_id"]) == set(sample["post_id"])
assert same_posts, "The sheet's posts don't match this notebook's sample. Did slang.csv change after the sheet was written?"

missing = labeled["label"].isna() | labeled["slang"].isna()
assert not missing.any(), f"{missing.sum()} posts still need a label or a slang flag."

bad_labels = labeled.loc[~labeled["label"].isin(LABELS), "label"].unique()
assert len(bad_labels) == 0, f"Labels must be one of {LABELS}. Found: {list(bad_labels)}"

bad_flags = labeled.loc[~labeled["slang"].isin([0, 1]), "slang"].unique()
assert len(bad_flags) == 0, f"Slang flags must be 1 or 0. Found: {list(bad_flags)}"

print(f"All {len(labeled)} posts have a valid label and slang flag.")

All 300 posts have a valid label and slang flag.


Join the author's tag back in by `post_id` and save the test set:

In [11]:
test = sample[["post_id", "text", "label", "auto_slang", "matched_terms"]].rename(columns={"label": "author_label"})
test["matched_terms"] = test["matched_terms"].str.join("; ")
test = test.merge(labeled[["post_id", "label", "slang"]], on="post_id")
test["slang"] = test["slang"].astype(int)
test = test[["post_id", "text", "label", "slang", "author_label", "auto_slang", "matched_terms"]]

test.to_csv(TEST_PATH, index=False)
print(f"Wrote {TEST_PATH.name}: {len(test)} posts")

Wrote test_hand_labeled.csv: 300 posts


**Group sizes and label mix.** The hand flag decides each post's group, and the hand label decides its class:

In [12]:
mix = pd.crosstab(test["slang"].map(GROUPS), test["label"], margins=True, margins_name="total")
mix.loc[["slang-heavy", "plain", "total"], LABELS + ["total"]]

label,bullish,bearish,neutral,total
slang,,,,
slang-heavy,46,31,72,149
plain,33,31,87,151
total,79,62,159,300


**Automatic vs. hand slang flag.** Disagreements are either literal matches the regex couldn't tell apart (automatic 1, hand 0) or slang that isn't on the list (automatic 0, hand 1).

In [13]:
agreement = (test["auto_slang"] == test["slang"]).mean()
print(f"The automatic flag agreed with the hand flag on {agreement:.0%} of posts.")
pd.crosstab(test["auto_slang"].map(GROUPS), test["slang"].map(GROUPS), rownames=["automatic flag"], colnames=["hand flag"])

The automatic flag agreed with the hand flag on 99% of posts.


hand flag,plain,slang-heavy
automatic flag,,
plain,149,1
slang-heavy,2,148


Posts where they disagreed (up to 6 shown):

In [14]:
disagree = test[test["auto_slang"] != test["slang"]]
for row in disagree.head(6).itertuples():
    print(f"auto {row.auto_slang}, hand {row.slang} | terms: {row.matched_terms or '-'} | {row.text[:90]}")

auto 1, hand 0 | terms: send it | let’s send it there
auto 1, hand 0 | terms: shill | this is literally a pyramid scheme tho and i'm not even a bot or paid shill. just don't li
auto 0, hand 1 | terms: - | staking this is the way to go if you like free money at 13%


## Summary

- The slang list has **118 terms**. About **7.0%** of the 1.25M cleaned posts contain at least one (7.7% of posts tagged bullish or bearish, 5.9% of untagged posts). The test set is half slang-heavy on purpose, so results are reported for each group separately.
- Drew **300 posts** from the held-out pool, 50 per group (automatic slang flag × author tag), and labeled them by hand without seeing the author tags.
- The hand flag gives **149 slang-heavy** and **151 plain** posts. The automatic flag agreed with it on **99%** of posts. Before the final run, a consistency check against the list's rules changed 6 hand flags: words like *bulls* and *bears* don't count as slang, and *whale* does.
- The hand labels are **79 bullish, 62 bearish and 159 neutral**. Over half are neutral even though the sample was balanced by author tag, a first sign that author tags often don't match a post's actual sentiment. Notebook 03 measures this properly. With only about 30 bearish posts per group, the bearish scores will be noisy.